**Universidade Federal do Pampa (Unipampa) — Campus Alegrete**  
AL5170 — Introdução à Ciência de Dados  
Prof. João Pablo S. da Silva

---

# Notebook 2 — Transformação e Engenharia de Variáveis

**Dataset:** Food Wastage Data in Restaurant  
**Fonte:** Kaggle — https://www.kaggle.com/datasets/trevinhannibal/food-wastage-data-in-restaurant  
**Grupo 3:** Arthur Provenzi Parizotto · Bianca Maia Ribeiro · Emilly Nascimento Dias · Rafael Jacques do Nascimento Lopes  
**Projeto:** Investigação do Desperdício de Alimentos em Serviços de Buffet

---

Este notebook documenta o processo de transformação do dataset silver em uma versão analítica (**gold**), preparada para responder às quatro questões investigáveis do projeto. Cada variável derivada e indicador é criado com propósito explícito, critério definido e vínculo direto com uma pergunta ou métrica do projeto.


## 1. Importação de Pacotes

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 2. Carregamento do Dataset Silver

Carrega a versão silver produzida pelo Notebook 1. Este é o ponto de entrada da etapa de transformação: os dados já passaram por limpeza, padronização de nomes, conversão de tipos categóricos e remoção de duplicatas completas.

Este notebook não lê novamente o CSV original do Kaggle. Sua entrada é exclusivamente `food_wastage_silver.csv`, preservando a sequência do pipeline: **fonte original → bronze → silver → gold**.


In [ ]:
df = pd.read_csv('food_wastage_silver.csv')

# Reconverte categorias (perdidas na serialização CSV)
cat_cols = ['type_of_food', 'event_type', 'storage_conditions',
            'purchase_history', 'seasonality', 'preparation_method',
            'geographical_location', 'pricing']
for col in cat_cols:
    df[col] = df[col].astype('category')

print(f"Dataset silver carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")
df.head()

## 3. Visão Geral das Variáveis Originais

Antes de criar variáveis derivadas, documenta-se o papel analítico de cada coluna original em relação às questões investigáveis do projeto.

In [ ]:
dicionario = pd.DataFrame({
    'Coluna': df.columns.tolist(),
    'Tipo': df.dtypes.astype(str).tolist(),
    'Questão Vinculada': [
        'Q1, Q2, Q3, Q4', 'Q2, Q4', 'Q2, Q4',
        'Q1, Q3, Q4', 'Q1', 'Q3',
        'Q3', 'Q1, Q4', '—', 'Q3, Q4', 'Q1, Q2, Q3, Q4'
    ]
})
print(dicionario.to_string(index=False))

## 3.1 Vínculo entre Perguntas Investigáveis e Transformações

O parecer solicita explicitar com mais precisão como as variáveis derivadas se conectam às perguntas investigáveis do projeto. A matriz abaixo registra, antes da engenharia de variáveis, qual pergunta cada transformação pretende apoiar.


In [ ]:
matriz_transformacoes = pd.DataFrame({
    'Pergunta': [
        'Q1', 'Q1', 'Q1',
        'Q2', 'Q2',
        'Q3', 'Q3',
        'Q4', 'Q4'
    ],
    'Objetivo analítico': [
        'Comparar desperdício entre métodos de preparo',
        'Comparar desperdício por condição de armazenamento',
        'Comparar taxa de desperdício por tipo de alimento',
        'Avaliar desperdício por convidado',
        'Comparar desperdício por porte do evento',
        'Relacionar quantidade produzida com taxa de desperdício',
        'Comparar taxa de desperdício por faixa de preço',
        'Construir perfis operacionais combinados',
        'Ranquear perfis pelo risco médio de desperdício'
    ],
    'Variável derivada ou métrica': [
        'mean_wastage_by_prep_method',
        'storage_inadequate',
        'mean_wastage_rate_by_food_type',
        'wastage_per_guest',
        'event_size',
        'quantity_tier',
        'pricing_ordered + wastage_rate',
        'event_profile',
        'mean_wastage_rate_by_profile + waste_risk_rank'
    ],
    'Por que é necessária': [
        'Permite comparar cada evento com a média de desperdício do seu método de preparo.',
        'Transforma a condição de armazenamento em indicador direto para comparação entre grupos.',
        'Normaliza o desperdício pelo total produzido e agrega por tipo de alimento.',
        'Remove o efeito do tamanho do público ao dividir desperdício por número de convidados.',
        'Segmenta eventos por número de convidados para comparar portes semelhantes.',
        'Agrupa a produção em faixas para observar tendência entre volume produzido e desperdício.',
        'Ordena as categorias de preço e permite comparar a taxa média de desperdício entre elas.',
        'Combina porte, preparo e preço para representar perfis operacionais completos.',
        'Ordena os perfis pela taxa média de desperdício para priorizar grupos de maior risco.'
    ]
})

print(matriz_transformacoes.to_string(index=False))


## 4. Transformações para Q1 — Método de Preparo, Armazenamento e Tipo de Alimento

**Q1:** O método de preparo e as condições de armazenamento influenciam o volume de desperdício?

As métricas de Q1 exigem o desperdício médio agrupado por `preparation_method` e `storage_conditions`, além da taxa de desperdício por `type_of_food`. Essas variáveis já existem na base; as transformações aqui criam indicadores calculados por grupo que responderão às hipóteses.

### 4.1 Taxa de Desperdício por Evento

**Fórmula:** `(wastage_food_amount / quantity_of_food) × 100`  
**Justificativa:** A quantidade bruta de desperdício não é comparável entre eventos com quantidades produzidas distintas. A taxa normaliza o desperdício pela produção, tornando eventos de portes diferentes comparáveis entre si. Esta variável é usada diretamente em Q1 (por método/armazenamento), Q3 e Q4.

In [ ]:
df['wastage_rate'] = (df['wastage_food_amount'] / df['quantity_of_food'] * 100).round(4)

print("=== Taxa de desperdício — Estatísticas ===")
print(df['wastage_rate'].describe().round(3))
print(f"\nInterpretação: Em média, {df['wastage_rate'].mean():.1f}% dos alimentos produzidos são desperdiçados.")

### 4.2 Variável de Armazenamento Binária

**Justificativa (H2 de Q1):** A hipótese H2 afirma que armazenamento inadequado está associado a maior desperdício. `storage_conditions` possui duas categorias: `'Refrigerated'` (adequado) e `'Room Temperature'` (potencialmente inadequado para alimentos perecíveis). Cria-se uma flag booleana para facilitar comparações binárias.

In [ ]:
df['storage_inadequate'] = (df['storage_conditions'] == 'Room Temperature')

print("=== Distribuição da flag de armazenamento inadequado ===")
print(df['storage_inadequate'].value_counts())
print(df.groupby('storage_inadequate')['wastage_rate'].mean().round(3))

### 4.3 Desperdício Médio por Método de Preparo e por Tipo de Alimento

Cria variáveis derivadas que associam a cada evento o desperdício médio do grupo ao qual ele pertence. Isso permite comparar cada evento com a média do seu grupo nas análises posteriores. Segue a fórmula da métrica: `Média(wastage_food_amount) agrupada por preparation_method`.

In [ ]:
# Desperdício médio por método de preparo (para cada evento)
mean_by_prep = df.groupby('preparation_method')['wastage_food_amount'].transform('mean')
df['mean_wastage_by_prep_method'] = mean_by_prep.round(3)

# Taxa média de desperdício por tipo de alimento
mean_rate_by_food = df.groupby('type_of_food')['wastage_rate'].transform('mean')
df['mean_wastage_rate_by_food_type'] = mean_rate_by_food.round(4)

print("=== Desperdício médio por método de preparo ===")
print(df.groupby('preparation_method')['wastage_food_amount'].mean().round(2))

print("\n=== Taxa média de desperdício por tipo de alimento ===")
print(df.groupby('type_of_food')['wastage_rate'].mean().round(3))

## 5. Transformações para Q2 — Tipo e Porte do Evento

**Q2:** Como o tipo e o porte do evento se relacionam com o volume de desperdício?

As hipóteses de Q2 exigem (H1) desperdício *per capita* e (H2) proporção de desperdício por tipo de evento. Ambas as métricas requerem variáveis derivadas não presentes na base silver.

### 5.1 Desperdício Per Capita

**Fórmula:** `wastage_food_amount / number_of_guests`  
**Justificativa (H1 de Q2):** A hipótese afirma que eventos maiores desperdiçam mais *por convidado*. Dividir o desperdício pelo número de convidados elimina o efeito do porte absoluto do evento, tornando a comparação justa entre eventos pequenos e grandes.

In [ ]:
df['wastage_per_guest'] = (df['wastage_food_amount'] / df['number_of_guests']).round(4)

print("=== Desperdício per capita — Estatísticas ===")
print(df['wastage_per_guest'].describe().round(4))

### 5.2 Faixa de Convidados (Porte do Evento)

**Justificativa:** A métrica de Q2 exige segmentar eventos por faixa de convidados para comparar o desperdício *per capita* por porte. A divisão em quartis garante que cada faixa tenha quantidade similar de eventos, evitando grupos com poucos registros.

- **Pequeno:** abaixo do 1º quartil (~267 convidados)  
- **Médio:** entre 1º e 3º quartil  
- **Grande:** acima do 3º quartil (~350 convidados)

In [ ]:
q1 = df['number_of_guests'].quantile(0.25)
q3 = df['number_of_guests'].quantile(0.75)

def classify_event_size(n):
    if n <= q1:
        return 'Pequeno'
    elif n <= q3:
        return 'Médio'
    else:
        return 'Grande'

df['event_size'] = df['number_of_guests'].apply(classify_event_size).astype('category')

print(f"Limites: Q1={q1:.0f}, Q3={q3:.0f}")
print("\n=== Distribuição de porte ===")
print(df['event_size'].value_counts())

print("\n=== Desperdício per capita por porte ===")
print(df.groupby('event_size', observed=True)['wastage_per_guest'].mean().round(4))

## 6. Transformações para Q3 — Quantidade Produzida, Preço e Taxa de Desperdício

**Q3:** Existe relação entre quantidade produzida, faixa de preço e taxa de desperdício?

A taxa de desperdício já foi criada (seção 4.1). As transformações de Q3 focalizam na categorização da quantidade produzida e no preparo das comparações por `pricing`.

### 6.1 Faixa de Quantidade Produzida

**Justificativa (H1 de Q3):** Para verificar se maior produção implica maior taxa de desperdício, é necessário categorizar a quantidade produzida em faixas comparáveis. A segmentação em tercis (baixo/médio/alto) distribui os eventos em grupos de tamanho similar, evitando grupos com poucos registros.

In [ ]:
t33 = df['quantity_of_food'].quantile(0.33)
t67 = df['quantity_of_food'].quantile(0.67)

def classify_qty(q):
    if q <= t33:
        return 'Baixa'
    elif q <= t67:
        return 'Média'
    else:
        return 'Alta'

df['quantity_tier'] = df['quantity_of_food'].apply(classify_qty).astype('category')

print(f"Tercis: T33={t33:.0f}, T67={t67:.0f}")
print("\n=== Distribuição de faixa de quantidade ===")
print(df['quantity_tier'].value_counts())

print("\n=== Taxa média de desperdício por faixa de quantidade ===")
print(df.groupby('quantity_tier', observed=True)['wastage_rate'].mean().round(3))

### 6.2 Taxa Média de Desperdício por Faixa de Preço

**Justificativa (H2 de Q3):** A hipótese afirma que eventos de preço mais alto apresentam maior taxa de desperdício. A variável `pricing` já possui três categorias (`Low`, `Moderate`, `High`). Criamos uma variável ordenada para garantir que análises posteriores respeitem a hierarquia de preço.

In [ ]:
pricing_order = ['Low', 'Moderate', 'High']
df['pricing_ordered'] = pd.Categorical(df['pricing'], categories=pricing_order, ordered=True)

print("=== Taxa média de desperdício por faixa de preço ===")
print(df.groupby('pricing_ordered', observed=True)['wastage_rate'].mean().round(3))

## 7. Transformações para Q4 — Combinações de Características Operacionais

**Q4:** Quais combinações de características operacionais estão associadas aos maiores níveis de desperdício?

Q4 é a questão mais integrativa: exige combinar porte (`event_size`), `preparation_method` e `pricing` em um perfil de evento único. As métricas de Q4 ranqueiam esses perfis pela taxa média de desperdício.

### 7.1 Perfil de Evento (Variável Composta)

**Fórmula da métrica:** taxa média de desperdício agrupada por `[event_size × preparation_method × pricing]`  
**Justificativa:** Cada combinação dessas três dimensões representa um *perfil operacional* distinto. Concatenar as categorias em uma única variável de texto permite agrupar, contar e ranquear perfis diretamente nas análises posteriores.

In [ ]:
df['event_profile'] = (
    df['event_size'].astype(str) + ' | ' +
    df['preparation_method'].astype(str) + ' | ' +
    df['pricing'].astype(str)
)

print(f"Total de perfis únicos: {df['event_profile'].nunique()}")
print("\n=== Distribuição por perfil (top 10) ===")
print(df['event_profile'].value_counts().head(10))

### 7.2 Ranking de Risco de Desperdício por Perfil

**Justificativa:** A métrica de Q4 exige ordenar os perfis pela taxa média de desperdício, produzindo um ranking de risco. Cada evento recebe o rank do perfil ao qual pertence, permitindo identificar diretamente os grupos de maior e menor risco.

In [ ]:
# Taxa média de desperdício por perfil
profile_risk = (
    df.groupby('event_profile')['wastage_rate']
    .mean()
    .round(4)
    .reset_index()
    .rename(columns={'wastage_rate': 'mean_wastage_rate_by_profile'})
    .sort_values('mean_wastage_rate_by_profile', ascending=False)
    .reset_index(drop=True)
)
profile_risk['waste_risk_rank'] = profile_risk.index + 1

# Merge de volta no DataFrame principal
df = df.merge(profile_risk[['event_profile', 'mean_wastage_rate_by_profile', 'waste_risk_rank']],
              on='event_profile', how='left')

print("=== Top 10 perfis com maior taxa média de desperdício ===")
print(profile_risk.head(10).to_string(index=False))

## 8. Seleção e Organização da Base Gold

A versão gold seleciona apenas as variáveis relevantes para responder às questões investigáveis, reorganizadas em ordem lógica: variáveis originais do evento, indicadores calculados e variáveis derivadas criadas neste notebook.

**Unidade de análise da camada gold:** cada linha da gold continua representando **um registro operacional/evento de desperdício** herdado da camada silver. A gold não é uma tabela agregada por categoria; ela é uma base enriquecida em nível de evento, na qual cada registro recebe métricas derivadas, classificações e rankings que permitem análises posteriores.

Variáveis auxiliares de cálculo intermediário, como `pricing_ordered`, são excluídas por redundância com `pricing`.


In [ ]:
gold_cols = [
    # Variáveis originais relevantes
    'type_of_food', 'event_type', 'number_of_guests',
    'quantity_of_food', 'wastage_food_amount',
    'preparation_method', 'storage_conditions',
    'purchase_history', 'seasonality',
    'geographical_location', 'pricing',
    # Variáveis derivadas — Q1
    'wastage_rate', 'storage_inadequate',
    'mean_wastage_by_prep_method', 'mean_wastage_rate_by_food_type',
    # Variáveis derivadas — Q2
    'wastage_per_guest', 'event_size',
    # Variáveis derivadas — Q3
    'quantity_tier',
    # Variáveis derivadas — Q4
    'event_profile', 'mean_wastage_rate_by_profile', 'waste_risk_rank'
]

df_gold = df[gold_cols].copy()

print(f"Dataset gold: {df_gold.shape[0]} linhas × {df_gold.shape[1]} colunas")
print("\nColunas da versão gold:")
for i, col in enumerate(df_gold.columns, 1):
    print(f"  {i:2d}. {col}")

## 9. Geração do Dataset Gold

In [ ]:
df_gold.to_csv('food_wastage_gold.csv', index=False)
print("Dataset gold gerado com sucesso.")
print(f"Dimensões finais: {df_gold.shape[0]} linhas × {df_gold.shape[1]} colunas")
df_gold.head()

## 10. Comparação Bronze × Silver × Gold

Evidencia a evolução do dataset ao longo das três camadas do pipeline, demonstrando como cada etapa agregou valor analítico.

In [ ]:
df_bronze = pd.read_csv('food_wastage_bronze.csv')
df_silver_ref = pd.read_csv('food_wastage_silver.csv')

comparison = pd.DataFrame({
    'Aspecto': ['Linhas', 'Colunas', 'Duplicatas', 'Nomes de colunas',
                'Variáveis derivadas', 'Indicadores calculados', 'Finalidade'],
    'Bronze': [
        df_bronze.shape[0], df_bronze.shape[1],
        df_bronze.duplicated().sum(),
        'Espaços / CamelCase', '0', '0',
        'Dado original materializado'
    ],
    'Silver': [
        df_silver_ref.shape[0], df_silver_ref.shape[1],
        df_silver_ref.duplicated().sum(),
        'snake_case', '0', '0',
        'Dado limpo'
    ],
    'Gold': [
        df_gold.shape[0], df_gold.shape[1],
        df_gold.duplicated().sum(),
        'snake_case', '10', '5',
        'Dado analítico em nível de evento'
    ]
})
print(comparison.to_string(index=False))


### Resumo das Variáveis Derivadas Criadas

In [ ]:
derivadas = pd.DataFrame({
    'Variável': [
        'wastage_rate', 'storage_inadequate',
        'mean_wastage_by_prep_method', 'mean_wastage_rate_by_food_type',
        'wastage_per_guest', 'event_size',
        'quantity_tier', 'event_profile',
        'mean_wastage_rate_by_profile', 'waste_risk_rank'
    ],
    'Origem': [
        'wastage / quantity × 100',
        'storage_conditions == Room Temp.',
        'mean(wastage) group by prep_method',
        'mean(wastage_rate) group by food_type',
        'wastage / number_of_guests',
        'quartis de number_of_guests',
        'tercis de quantity_of_food',
        'event_size | prep_method | pricing',
        'mean(wastage_rate) por perfil',
        'rank decrescente por perfil'
    ],
    'Q': ['Q1,Q3,Q4','Q1','Q1','Q1','Q2','Q2,Q4','Q3','Q4','Q4','Q4']
})
print(derivadas.to_string(index=False))

## 11. Limitações da Transformação Realizada

**1. Gold em nível de evento, não agregada:** a camada gold preserva a granularidade da silver. Cada linha é um registro operacional/evento enriquecido por variáveis analíticas. Rankings e médias por grupo aparecem como atributos derivados, mas a tabela final não substitui análises agregadas específicas.

**2. Faixas definidas por quartis/tercis:** os limites de `event_size` e `quantity_tier` são estatísticos e dependem da distribuição deste dataset, não de critérios operacionais externos. Se novos dados forem adicionados, os limites podem mudar.

**3. Médias por grupo calculadas no próprio conjunto:** variáveis como `mean_wastage_by_prep_method` e `mean_wastage_rate_by_profile` foram calculadas sobre toda a silver. Para modelagem preditiva futura, essas métricas devem ser recalculadas apenas no conjunto de treino para evitar vazamento de informação.

**4. Perfis com poucos registros:** algumas combinações de `event_size × preparation_method × pricing` podem ter poucos eventos. Nesses casos, a taxa média e o ranking de risco podem ser instáveis e devem ser interpretados com cautela.

**5. Taxa de desperdício sem ponderação econômica:** `wastage_rate` mede proporção física de desperdício, mas não considera custo, valor nutricional ou impacto ambiental específico de cada tipo de alimento.

**6. Limitações herdadas da fonte:** a gold melhora a capacidade analítica, mas não resolve limitações do dataset original, como ausência de data, ausência de identificador único, unidade não documentada das quantidades e representatividade não comprovada.
